# Stage 1: Supervised Fine-Tuning (SFT) on GPT-2 (124M)

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from transformers.trainer_utils import get_last_checkpoint
import os

# ---------------------------
# Config – keep these exact values for reproducibility
# ---------------------------
MODEL_NAME = "openai-community/gpt2"
OUTPUT_DIR = "./sft_gpt2_dolly"
MAX_SEQ_LEN = 256          # 512 also possible but tighter on T4
NUM_EPOCHS = 2
PER_DEVICE_BATCH = 8       # safe on T4; raise to 16 if memory allows
GRAD_ACCUM = 8             # effective batch ≈ 64
LR = 5e-5
FP16 = True
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# 1. Load tokenizer & model
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

# ---------------------------
# 2. Load & format Dolly-15k
# ---------------------------
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

def format_example(example):
    # Simple clean format (no special tokens required, but easy to add)
    instruction = example["instruction"].strip()
    context = example.get("context", "").strip()
    response = example["response"].strip()

    if context:
        prompt = f"Instruction: {instruction}\nContext: {context}\nResponse:"
    else:
        prompt = f"Instruction: {instruction}\nResponse:"

    full_text = f"{prompt} {response}{tokenizer.eos_token}"
    return {"text": full_text}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

# Train / validation split
dataset = dataset.train_test_split(test_size=0.1, seed=SEED)
train_ds = dataset["train"]
val_ds = dataset["test"]

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,          # dynamic padding later
    )

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["text"])

# ---------------------------
# 3. Data collator
# ---------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,                 # causal LM
)

# ---------------------------
# 4. Training arguments
# ---------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=FP16,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",          # disable wandb etc. for minimal run
    seed=SEED,
    dataloader_pin_memory=False,  # sometimes helps on Colab
)

# ---------------------------
# 5. Trainer
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

# ---------------------------
# 6. Train
# ---------------------------
print("Starting SFT training...")
print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")
print(f"Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"Max sequence length: {MAX_SEQ_LEN}")

trainer.train()

# ---------------------------
# 7. Save final checkpoint
# ---------------------------
final_path = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"\nSFT checkpoint saved to: {final_path}")

# Quick sanity check – generate a few samples
print("\n--- Quick generation check ---")
model.eval()
test_prompts = [
    "Instruction: Explain what a black hole is in simple terms.\nResponse:",
    "Instruction: Write a short poem about the ocean.\nResponse:",
]
for p in test_prompts:
    inputs = tokenizer(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    print(tokenizer.decode(out[0], skip_special_tokens=True))
    print("-" * 60)